- **MapType**: 하나의 Column안에 **Key-Value** 쌍 구조로 데이터를 저장함 (Sub-column 구조와 유사)

In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.master('local[*]').appName('11').getOrCreate()

In [14]:
# tuple을 요소로 하는 list
# tuple이 각 행 데이터를 의미함
map_data = [
    ('Tom',{'height':173.5,'weight':69.8}),
    ('Jane',{'height':163.0,'weight':45.5,'blood_p':110.0})
]

In [15]:
map_schema = ['name','properties']

- `createDataFrame`의 `schema`는 `StructType`을 포함한 `DataType`을 지정할 수 있으나, `string`을 요소로 하는 `list`를 `schema`로 전달하는 경우, 단순히 **Column명**으로만 지정되며, Column의 데이터타입은 자동으로 찾음
- `StructType` -> `StructField` -> 각 column의 이름, 데이터타입, nullable 지정가능

In [16]:
df = spark.createDataFrame(map_data,schema = map_schema)
# map_schema list가 포함하고 있는 문자열의 컬럼을 생성하라는 의미

In [17]:
df.show(truncate=False)

+----+---------------------------------------------------+
|name|properties                                         |
+----+---------------------------------------------------+
|Tom |{weight -> 69.8, height -> 173.5}                  |
|Jane|{weight -> 45.5, blood_p -> 110.0, height -> 163.0}|
+----+---------------------------------------------------+



In [18]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: double (valueContainsNull = true)



- Data가 `dict` 형태로 로드되면, **MapType**으로 지정될 수 있음
- **MapType**의 특성
> - 반드시 Key와 Value 각각의 타입은 통일되어야 함 (위의 경우, Key는 str, Value는 double)
> - 하나라도 일치하지 않으면, **Null, type 변환, 오류 발생** 등의 문제가 발생할 수 있음
> - **왜??**: 성능최적화 및 효율성 (모두 동일한 타입인 경우 , 빠르게 데이터를 찾을 수 있으나, 데이터 타입이 다른 경우, 찾는 속도도 떨어지며, 어떤 데이터타입인지도 기록해야 함)

In [19]:
# MapType의 value 접근
# height, weight, blood_p 열을 생성하여, 각 value를 할당함
df.withColumn('height',df.properties.height).\
withColumn('weight',df['properties']['weight']).\
withColumn('blood_p',df.properties.getItem('blood_p')).show()
# df.properties.height, df['properties']['weight']
# df.properties.['weight'], df['properties'].weight 모두 가능

+----+--------------------+------+------+-------+
|name|          properties|height|weight|blood_p|
+----+--------------------+------+------+-------+
| Tom|{weight -> 69.8, ...| 173.5|  69.8|   NULL|
|Jane|{weight -> 45.5, ...| 163.0|  45.5|  110.0|
+----+--------------------+------+------+-------+



- `createDataFrame`에서 schema를 `StructType`으로 지정
- `MapType`은 Key와 Value의 타입을 각각 지정해야 함

In [20]:
struct_schema = StructType([
    StructField('name',StringType(),True),
    StructField('properties',MapType(StringType(),FloatType()))
])

In [21]:
df_struct = spark.createDataFrame(map_data,schema=struct_schema)
df_struct.show(truncate=False)
df_struct.printSchema()

+----+---------------------------------------------------+
|name|properties                                         |
+----+---------------------------------------------------+
|Tom |{weight -> 69.8, height -> 173.5}                  |
|Jane|{weight -> 45.5, blood_p -> 110.0, height -> 163.0}|
+----+---------------------------------------------------+

root
 |-- name: string (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: float (valueContainsNull = true)

